## Tâche 8 — Le verdict

Je touche au jeu de test pour la première fois. 19 911 réservations, de
septembre à décembre 2016, que ni la v1 rejouée ni la v2 n'ont jamais vues.

Avant de regarder, je choisis ma métrique.

Une annulation ratée, c'est une chambre qui reste bloquée et qui part vide.
L'hôtel perd la nuitée.

Une annulation prédite à tort, c'est une chambre remise en vente alors que le
client vient. Il faut le reloger ailleurs, et il ne revient pas.

Les deux coûtent, et aucune ne domine franchement. Je prends le F1, qui tombe
dès que l'un des deux côtés se dégrade. Je regarde aussi le coût en euros, pour
vérifier que mon choix ne tient pas qu'à une formule.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

query = text("""
    SELECT r.*, h.code AS hotel
    FROM reservations r
    JOIN hotels h ON r.hotel_id = h.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date", "reservation_status_date"])

TEST_MONTHS = 4
debut_test = (df["arrival_date"].max().to_period("M") - (TEST_MONTHS - 1)).to_timestamp()

train = df[df["arrival_date"] < debut_test].copy()
test = df[df["arrival_date"] >= debut_test].copy()

print(f"train : {len(train)} | test : {len(test)}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

CATEGORICAL = ["hotel", "arrival_date_month", "market_segment", "deposit_type",
               "customer_type", "country", "meal", "distribution_channel",
               "reserved_room_type"]

NUMERICAL = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
             "adults", "children", "babies", "adr", "previous_cancellations",
             "previous_bookings_not_canceled", "total_of_special_requests",
             "required_car_parking_spaces", "is_repeated_guest", "days_in_waiting_list",
             "duree_sejour", "taille_groupe", "avec_enfants", "taux_annulation_passe"]


def ajouter_variables(donnees):
    d = donnees.copy()
    enfants = d["children"].fillna(0) + d["babies"]
    d["duree_sejour"] = d["stays_in_weekend_nights"] + d["stays_in_week_nights"]
    d["taille_groupe"] = d["adults"] + enfants
    d["avec_enfants"] = (enfants > 0).astype(int)
    historique = d["previous_cancellations"] + d["previous_bookings_not_canceled"]
    d["taux_annulation_passe"] = np.where(
        historique > 0,
        d["previous_cancellations"] / historique.replace(0, np.nan),
        0.0,
    )
    return d[CATEGORICAL + NUMERICAL]


def build_pipeline_v2(classifier, mise_a_l_echelle=False):
    etapes_num = [("impute", SimpleImputer(strategy="median"))]
    if mise_a_l_echelle:
        etapes_num.append(("scale", StandardScaler()))

    preprocessing = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
            ]), CATEGORICAL),
            ("num", Pipeline(etapes_num), NUMERICAL),
        ],
        sparse_threshold=0,
    )

    return Pipeline([
        ("variables", FunctionTransformer(ajouter_variables)),
        ("preprocessing", preprocessing),
        ("classifier", classifier),
    ])

### Entraîner les deux modèles et mesurer

In [ ]:
import joblib
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from model.train_baseline import FEATURES, TARGET, build_pipeline


def foret():
    """Une forêt neuve avec les réglages trouvés en tâche 7."""
    return RandomForestClassifier(n_estimators=200, min_samples_leaf=100,
                                  max_features=0.7, random_state=42, n_jobs=-1)


y_test = test[TARGET]
resultats, matrices = [], {}
modeles, probas = {}, {}          # pour garder chaque modèle et ses probabilités

candidats = [
    ("v1 rejouee", build_pipeline(), train[FEATURES], test[FEATURES]),
    ("v2", build_pipeline_v2(foret()), train.drop(columns=[TARGET]), test.drop(columns=[TARGET])),
]

for nom, pipe, X_tr, X_te in candidats:
    debut = time.perf_counter()
    pipe.fit(X_tr, train[TARGET])             # entraînement sur le train uniquement
    duree = time.perf_counter() - debut

    pred = pipe.predict(X_te)                 # 0 ou 1, au seuil 0,5 par défaut
    proba = pipe.predict_proba(X_te)[:, 1]    # probabilité d'annulation

    modeles[nom] = pipe
    probas[nom] = proba

    chemin = ROOT / "model" / f"mesure_{nom.split()[0]}.joblib"
    joblib.dump(pipe, chemin)                 # pour mesurer la taille sur disque

    matrices[nom] = confusion_matrix(y_test, pred)
    resultats.append({
        "modele": nom,
        "f1": round(f1_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "roc_auc": round(roc_auc_score(y_test, proba), 4),
        "entrainement_s": round(duree, 1),
        "taille_Mo": round(chemin.stat().st_size / 1e6, 1),
    })

pd.DataFrame(resultats)

### Les matrices de confusion

In [ ]:
for nom, m in matrices.items():
    tn, fp, fn, tp = m.ravel()
    print(f"\n{nom}")
    print(pd.DataFrame(m,
                       index=["réel : vient", "réel : annule"],
                       columns=["prédit : vient", "prédit : annule"]))
    print(f"  fausses alertes : {fp}  |  annulations ratées : {fn}")

In [ ]:
# from_estimator applique toujours le seuil 0,5, il appelle predict() lui-même
ConfusionMatrixDisplay.from_estimator(modeles["v1 rejouee"], test[FEATURES], y_test,
                                      display_labels=["maintenue", "annulée"]);

### Le coût en euros

In [ ]:
# une annulation ratée coûte la valeur de la réservation, une fausse alerte en coûte le double
nuits = (test["stays_in_weekend_nights"] + test["stays_in_week_nights"]).clip(lower=1)
valeur_moyenne = (test["adr"] * nuits).mean()

for nom, m in matrices.items():
    tn, fp, fn, tp = m.ravel()
    cout = fn * valeur_moyenne + fp * 2 * valeur_moyenne
    print(f"{nom:12} {cout:>12,.0f} EUR")

### Le seuil de décision

predict() tranche à 0,5. Mais seules 35,6 % des réservations sont annulées, donc
exiger plus d'une chance sur deux revient à n'en signaler presque aucune.

Je choisis le seuil sur les 4 derniers mois du jeu d'entraînement. Le modèle
n'apprend pas sur ces mois-là, et le test n'est jamais touché.

In [ ]:
def decouper_derniers_mois(donnees, nb_mois):
    """Sépare un jeu en deux : avant / après les nb_mois derniers mois d'arrivée."""
    bascule = (donnees["arrival_date"].max().to_period("M") - (nb_mois - 1)).to_timestamp()
    return (donnees[donnees["arrival_date"] < bascule].copy(),
            donnees[donnees["arrival_date"] >= bascule].copy())


def choisir_seuil(donnees, construire, colonnes=None, nb_mois=4):
    """Choisit le seuil qui maximise le F1 sur les derniers mois du jeu d'entraînement."""
    ajuste, valide = decouper_derniers_mois(donnees, nb_mois)

    X_ajuste = ajuste[colonnes] if colonnes else ajuste.drop(columns=[TARGET])
    X_valide = valide[colonnes] if colonnes else valide.drop(columns=[TARGET])

    modele = construire().fit(X_ajuste, ajuste[TARGET])
    proba = modele.predict_proba(X_valide)[:, 1]

    lignes = []
    for s in np.round(np.arange(0.15, 0.71, 0.05), 2):
        pred = (proba >= s).astype(int)
        lignes.append({
            "seuil": s,
            "f1": f1_score(valide[TARGET], pred),
            "precision": precision_score(valide[TARGET], pred, zero_division=0),
            "rappel": recall_score(valide[TARGET], pred),
        })

    trace = pd.DataFrame(lignes).round(3)
    return float(trace.loc[trace["f1"].idxmax(), "seuil"]), trace


seuil_v2, trace_v2 = choisir_seuil(train, lambda: build_pipeline_v2(foret()))
print("seuil retenu pour la v2 :", seuil_v2)
trace_v2

In [ ]:
seuil_v1, trace_v1 = choisir_seuil(train, build_pipeline, FEATURES)
print("seuil retenu pour la v1 :", seuil_v1)
trace_v1

### Réévaluation sur le test avec les seuils retenus

In [ ]:
SEUILS = {"v1 rejouee": seuil_v1, "v2": seuil_v2}

comparaison = []
for nom in ["v1 rejouee", "v2"]:
    for etiquette, seuil in [("0,50 par défaut", 0.50), (f"{SEUILS[nom]:.2f} retenu", SEUILS[nom])]:
        pred = (probas[nom] >= seuil).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        comparaison.append({
            "modele": nom,
            "seuil": etiquette,
            "f1": round(f1_score(y_test, pred), 4),
            "precision": round(precision_score(y_test, pred), 4),
            "recall": round(recall_score(y_test, pred), 4),
            "fausses_alertes": fp,
            "annulations_ratees": fn,
            "cout_EUR": round(fn * valeur_moyenne + fp * 2 * valeur_moyenne),
        })

pd.DataFrame(comparaison)

In [ ]:
# from_predictions, parce que predict() ignore le seuil choisi
pred_v2 = (probas["v2"] >= seuil_v2).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, pred_v2,
                                        display_labels=["maintenue", "annulée"]);

### Ce que je retiens

La v2 gagne, quel que soit le seuil. 0,68 contre 0,58 à 0,50, et 0,73 contre 0,68
une fois le seuil choisi.

La v1 a une précision de 0,92, elle ne se trompe presque jamais quand elle annonce
une annulation. Mais elle en rate 58 %. Prudente au point d'être inutile.

Le seuil 0,5 est un mauvais réglage par défaut. En le descendant à 0,30, la v2
rattrape 841 annulations de plus sur le test.

J'ai vérifié que mon choix ne tient pas à mon hypothèse de coût. Il faudrait
qu'une fausse alerte coûte plus de 3,3 fois une annulation ratée pour que la v1
repasse devant. Reloger un client ne coûte pas trois nuitées et demie.

Une tension reste. Le seuil qui maximise le F1 est 0,30, celui qui minimise mon
coût en euros est 0,50. C'est logique : le F1 compte une fausse alerte et une
annulation ratée pareil, mon calcul de coût non. Je garde 0,30, parce qu'on ne
revend pas une chambre sur la foi d'un modèle à 81 % de précision. On relance le
client, et ça ne coûte qu'un e-mail.

Les deux modèles chutent sur le test, 9 points pour la v2, 12 pour la v1. Normal :
le test est l'automne 2016, que ni l'un ni l'autre n'a vu, et il y a 38 %
d'annulations contre 35,6 % sur l'entraînement. Ma validation croisée était trop
optimiste. Le TimeSeriesSplit de la tâche 7 l'avait annoncé.